# One-day setups: pre-registration (Phase 13, step 2)

This notebook contains **only a pre-registration**: two new long-only setups for one-day holds on BTC/USDT, the development-period thresholds they use, development-period frequency counts, and the evaluation plan.

**No outcome has been computed.** No return of any trade or day after a signal, no outcome statistic, and no relationship between any condition and price direction appears here or was computed before writing it. The numbers computed are:
- one quantile of a signal-day volatility measure over the 2017–2023 development period;
- counts of how many development days each setup would trade.

The existing continuation and pullback setups are not reused.

**What the reasons may draw on.** Notebooks 10–14 measured volatility and counts only:
- **The cost is small against a one-day move:** 0.30% round trip was 0.085 of a typical one-day move overall (typical move 3.52%), and 0.109–0.138 in 2023–2026 (notebook 14).
- **Saturday entries are the calmest days:**
  - in 2017–2023 the typical Saturday move was 2.72%, the lowest of any weekday (cost ratio 0.110, against 0.059–0.098 on other days);
  - in 2024 onward it was 1.23% (cost ratio 0.244, against 0.106–0.147).
- **Active US opens are followed by larger moves.** Days after a highly active US open (09:00–10:59 New York) had larger one-day moves: in 2017–2023 the high-activity tercile had a typical move of 4.68% (cost ratio 0.064) against 2.78% for the low tercile (0.108); 2024 onward, 2.97% (0.101) against 2.24% (0.134).
- **Intraday timing:** 1-hour moves are largest around the US open (notebooks 11–12).

No one-day price-direction result exists, so the reasons below rest on these facts and on general market reasoning.

## Common rules (both setups)

**Trades.**
- A signal at the daily close of day t (UTC) buys at the open of day t + 1 and sells at the open of day t + 2.
- Trades come from `find_trades(signal, 1)` in `cryptosense.backtest`, unchanged: one position at a time. A signal on day t + 1, while the trade from day t is still open, is ignored, even though that trade exits at the open where the next one would enter.
- Long only, all-in, no leverage.

**Costs.** 0.30% round trip: 0.15% per side (0.10% fee + 0.05% slippage). The net log return of a trade is log(exit × (1 − 0.0015) / (entry × (1 + 0.0015))), the same cost convention as `run_backtest`.

**Signal-day US-open measures** (from 15-minute candles, on the UTC day t):
- The window is the 8 15-minute candles opening from 09:00 to 10:45 New York time (13:00–14:45 UTC in US summer time, 14:00–15:45 UTC otherwise). It ends at 11:00 New York, before the daily close of day t, so it is known at the signal.
- Each candle's 15-minute log return is its close against the previous candle's close, as in `add_intraday_features`.
- `rv_open` = square root of the sum of the 8 squared returns: how active the US open was.
- `ret_open` = sum of the 8 returns = log(close at 11:00 New York / close at 09:00 New York): the net price change over the US open.
- If any of the 8 returns is missing (a missing candle, including the 08:45 candle the first return needs), both measures are NaN and day t cannot signal.

**Daily inputs.** `add_regimes(add_features(...))` on daily data: `trend` and `log_return` (close of day t against close of day t − 1). A day with a missing input never signals.

**Saturday entries are excluded in both setups** (signals at the close of a Friday, UTC, are not traded).
- In the development period, Saturday was the calmest entry day (typical move 2.72%, against 3.06–5.09% on other days), so a Saturday trade has the least room to cover the 0.30% cost.
- Both setups rest on activity from the US session, which is closed at the weekend, so the mechanism they assume is weakest on Saturday.
- Sunday (the second-calmest day) is not excluded. Only the clear outlier is removed, to avoid stacking filters.
- 2024 onward showed the same ordering (Saturday 1.23%). The exclusion is justified by the development figures alone.

**Threshold.** Both setups use one quantile-based threshold: the 2/3 quantile of `rv_open` over development signal days.
- Development signal days are days t whose trade would exit (day t + 2) on or before 2023-12-31, with a complete window.
- All weekdays are included in the quantile, so it matches the terciles of notebook 14.
- "High activity" means the top third: a conventional split, and the one used in notebook 14's exploration. No other quantile was tried.

The threshold is computed and printed below.

## Development-period threshold

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from cryptosense.data import load_processed
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes
from cryptosense.intraday import NY_TZ, lag_by_time
from cryptosense.backtest import find_trades

DATA_DIR = Path.cwd().parent / "data"
SYMBOL = "BTC/USDT"
DEV_LAST_EXIT = pd.Timestamp("2023-12-31", tz="UTC")
WINDOW_HOURS_NY = [9, 10]
CANDLES_IN_WINDOW = 8
EXCLUDED_ENTRY_WEEKDAY = "Saturday"

daily = add_regimes(add_features(load_processed(SYMBOL, DATA_DIR)))
m15 = load_processed(SYMBOL, DATA_DIR, timeframe="15m")

# Signal-day US-open measures from the 8 window returns (NaN unless all 8 exist).
log_close = np.log(m15["close"])
ret_15m = log_close - lag_by_time(log_close, "15min")
in_window = np.isin(m15.index.tz_convert(NY_TZ).hour, WINDOW_HOURS_NY)
w = pd.DataFrame({"r": ret_15m[in_window], "day": m15.index[in_window].normalize()})
per_day = w.groupby("day")["r"].agg(
    rv_open=lambda r: np.sqrt((r ** 2).sum()), ret_open="sum", returns="count"
)
complete = per_day["returns"] == CANDLES_IN_WINDOW
per_day.loc[~complete, ["rv_open", "ret_open"]] = np.nan
per_day.index = per_day.index.as_unit(daily.index.unit)
daily = daily.join(per_day[["rv_open", "ret_open"]])

# Exit day (t + 2) and entry weekday (t + 1) for every possible signal day t.
daily["exit_day"] = pd.Series(daily.index, index=daily.index).shift(-2)
daily["entry_weekday"] = pd.Series(daily.index + pd.Timedelta(days=1), index=daily.index).dt.day_name()

dev_signal_days = daily[daily["exit_day"] <= DEV_LAST_EXIT]
dev_rv = dev_signal_days["rv_open"].dropna()
RV_OPEN_HIGH = dev_rv.quantile(2 / 3)

print(f"development signal days: {dev_signal_days.index[0].date()} -> {dev_signal_days.index[-1].date()} "
      f"({len(dev_signal_days):,} days; {len(dev_rv):,} with a complete US-open window)")
print(f"rv_open quantiles (development): 1/3 = {dev_rv.quantile(1 / 3):.6f}, 2/3 = {RV_OPEN_HIGH:.6f}")
print(f"\nThreshold used by both setups: rv_open > {RV_OPEN_HIGH:.6f}")

development signal days: 2017-08-17 -> 2023-12-29 (2,326 days; 2,320 with a complete US-open window)
rv_open quantiles (development): 1/3 = 0.005761, 2/3 = 0.010688

Threshold used by both setups: rv_open > 0.010688


## Setup D1 — Strong, active US open in an uptrend

**Idea.** In a daily uptrend, when the US open on day t was highly active and price rose over it, buy at the next day's open and hold for one day.

**Why it might work.**
- **Room to cover costs:** days after a highly active US open had the largest one-day moves in notebook 14, so a trade there has the most room to cover the 0.30% cost.
- **Persistent flows (general market reasoning):** a rise during the most active part of the trading day reflects buying by US participants, such as funds and ETF flows. Such flows are often spread over more than one session, so they may continue into the next day.
- **Trend filter:** restricting to days in a daily uptrend keeps long trades on the side of the prevailing trend.

This is a hypothesis; no one-day price-direction result supports or contradicts it.

**Conditions** (all on signal day t):
1. `trend == "up"`
2. `rv_open > RV_OPEN_HIGH`: top third of development US-open activity.
3. `ret_open > 0`: price rose from 09:00 to 11:00 New York.
4. The entry day (t + 1) is not a Saturday.

**Threshold justification.** `ret_open > 0` is a sign condition. `RV_OPEN_HIGH` is the development 2/3 quantile printed above. `trend` is the existing daily regime label.

## Setup D2 — Active US-open sell-off recovered by the close

**Idea.** When the US open on day t was highly active and price fell over it, but day t still closed above the previous day's close, buy at the next day's open and hold for one day.

**Why it might work.**
- **Room to cover costs:** as for D1, a highly active US open marks days followed by larger one-day moves.
- **Selling absorbed (general market reasoning):** a sharp, active sell-off at the US open that is fully recovered by the close suggests the selling was absorbed by buyers during the rest of the session. A day that closes up after an early sell-off is often read as a sign of underlying demand.
- **No trend filter:** the condition describes the day itself, so D2 is distinct from the existing pullback setup.

This is a hypothesis; no one-day price-direction result supports or contradicts it.

**Conditions** (all on signal day t):
1. `rv_open > RV_OPEN_HIGH`: top third of development US-open activity.
2. `ret_open < 0`: price fell from 09:00 to 11:00 New York.
3. `log_return > 0`: day t closed above the close of day t − 1.
4. The entry day (t + 1) is not a Saturday.

**Threshold justification.** `ret_open < 0` and `log_return > 0` are sign conditions. `RV_OPEN_HIGH` is the same development 2/3 quantile.

**Overlap.** D1 requires `ret_open > 0` and D2 requires `ret_open < 0`, so they never signal on the same day.

## Development-period frequency counts (counts only, no returns)

For each setup:
- **Signal days:** development days on which every condition holds.
- **Trades:** the trades `find_trades(signal, 1)` makes from those signal days.
- **Skipped for missing data:** non-Saturday-entry days whose window is incomplete or that have a missing input.

Development = trades exiting on or before 2023-12-31. No price after a signal is used.

In [2]:
SETUPS_D = {
    "D1": {"inputs": ["trend", "rv_open", "ret_open"],
           "rule": lambda d: (d["trend"] == "up") & (d["rv_open"] > RV_OPEN_HIGH) & (d["ret_open"] > 0)},
    "D2": {"inputs": ["rv_open", "ret_open", "log_return"],
           "rule": lambda d: (d["rv_open"] > RV_OPEN_HIGH) & (d["ret_open"] < 0) & (d["log_return"] > 0)},
}

not_saturday = daily["entry_weekday"] != EXCLUDED_ENTRY_WEEKDAY
rows = []
for name, setup in SETUPS_D.items():
    usable = daily[setup["inputs"]].notna().all(axis=1)
    signal = (usable & setup["rule"](daily).fillna(False).astype(bool) & not_saturday)
    trades = find_trades(signal, 1)
    trade_signal_days = daily.index[[entry - 1 for entry, _ in trades]]
    dev_mask = daily["exit_day"] <= DEV_LAST_EXIT
    dev_trades = [d for d in trade_signal_days if daily.loc[d, "exit_day"] <= DEV_LAST_EXIT]
    rows.append({
        "setup": name,
        "development days": int(dev_mask.sum()),
        "excluded (Saturday entry)": int((dev_mask & ~not_saturday).sum()),
        "skipped (missing data)": int((dev_mask & not_saturday & ~usable).sum()),
        "signal days": int((dev_mask & signal).sum()),
        "trades": len(dev_trades),
    })
frequency = pd.DataFrame(rows).set_index("setup")
print(frequency.to_string())

       development days  excluded (Saturday entry)  skipped (missing data)  signal days  trades
setup                                                                                          
D1                 2326                        333                      90          103      84
D2                 2326                        333                       6          122     110


## Evaluation plan (not run)

Nothing below is computed until this pre-registration is approved. After approval, the BTC development evaluation is run first. The held-out test and the ETH/SOL replication are then each run **once**, with exactly these rules, thresholds, measures and baselines, and are never re-run after changes.

### Periods
- **Development:** trades whose exit day (t + 2) is on or before 2023-12-31.
- **Held-out test:** trades whose entry day (t + 1) is on or after 2024-01-01.
- The single trade entering on 2023-12-31 and exiting on 2024-01-01 belongs to neither period.
- `RV_OPEN_HIGH` stays at the development value above for the test period.

### Measures
Per setup and period, on the net one-day log return after 0.30% round-trip costs:
- number of trades, signal days, and days skipped for missing data;
- mean, median, share of trades with a positive net return;
- standard error of the mean (sample standard deviation / √n).

### Base rates
In the same period and at the same cost, every day t is treated as a one-day trade (buy at the open of t + 1, sell at the open of t + 2), with Saturday entries excluded as in the setups. Each base rate is compared with the setup mean as (setup mean − base-rate mean) / setup standard error.
1. **All days.** The mean over every eligible day.
2. **Same entry weekday.** The mean for each entry weekday, combined with weights equal to the setup's share of trades on each weekday.
3. **Same entry weekday and same previous-day trend regime.** The mean in each (entry weekday, `trend` on day t) cell, combined with weights equal to the setup's share of trades in each cell.
   - "Previous-day" means the day before entry, i.e. the signal day t.
   - Days without a `trend` label are excluded from this base rate.
   - Its weights come from the setup's trades that have a label. Every D1 trade has one; D2 trades before the label exists are counted and reported.

### Random-entry baseline matched by weekday
- For each setup and period, draw without replacement, from the eligible base-rate days of each entry weekday, as many days as the setup traded on that weekday, and record the mean net return.
- Repeat 2,000 times, with seed 0 for development and seed 1 for test (one generator per period, used for D1 then D2).
- Report the median of the random means and the share of random means greater than or equal to the setup's mean.

### ETH/SOL replication (unchanged rules)
- After the BTC held-out run, D1 and D2 are applied unchanged to ETH/USDT and SOL/USDT: the same conditions, the same Saturday exclusion, periods, costs, measures, base rates and random baseline.
- **15-minute data needed:** the US-open measures need 15-minute ETH and SOL data, not yet downloaded. It is downloaded once with `refresh_symbol(..., timeframe="15m")` before the replication, with no other change.
- **Threshold per asset:** `rv_open` is not scale-free (ETH and SOL are more volatile than BTC). The rule "above the 2/3 quantile of development US-open activity" is therefore applied with each asset's own development-period quantile, computed from its own 2017–2023 data (SOL from its first available data) before any outcome. All other conditions are sign conditions or regime labels and carry over as they are.

### Number of comparisons, stated explicitly
- **BTC:** 2 setups × (3 base rates + 1 random baseline) × 2 periods = **16** (8 development, 8 held-out).
- **ETH/SOL replication:** 2 assets × 2 setups × 4 × 2 periods = **32**.
- **Total: 48 comparisons.**
- As a reference only (not a pass/fail rule): for the 8 BTC held-out comparisons, a Bonferroni-adjusted two-sided 5% level corresponds to |difference / SE| > 2.73.

### Limitations known in advance
- **The held-out test is not fully untouched.** 2024 onward daily BTC data was already examined in Phases 4–6, and its volatility in notebooks 10–14. Its volatility figures were consistent with the Saturday exclusion, which is justified by development figures alone. Replication on ETH and SOL is the main independent check.
- **The US-open window was chosen in the intraday track** using volatility statistics that included 2024 onward (Gates 1–2).
- **Trade counts may be small:** about a third of days pass the activity filter, before the direction and trend conditions. Standard errors are reported for that reason.
- **`find_trades` with a one-day hold skips a signal on the day after a traded signal.** The trade counts above reflect that.
- **Costs:** the 0.15% per side assumption should be checked against the current exchange fee schedule.

## Updated frequency counts after the rule change (counts only, no returns)

Rule change approved before any outcome: every signal day is its own trade, built with `find_one_day_trades` (`cryptosense.backtest`) instead of `find_trades(signal, 1)`. The development counts above are recomputed with it below. No price after a signal is used.

In [3]:
from cryptosense.backtest import find_one_day_trades

rows = []
for name, setup in SETUPS_D.items():
    usable = daily[setup["inputs"]].notna().all(axis=1)
    signal = usable & setup["rule"](daily).fillna(False).astype(bool) & not_saturday
    trade_signal_days = daily.index[[entry - 1 for entry, _ in find_one_day_trades(signal)]]
    dev_mask = daily["exit_day"] <= DEV_LAST_EXIT
    rows.append({
        "setup": name,
        "signal days": int((dev_mask & signal).sum()),
        "trades (find_trades, superseded)": int(frequency.loc[name, "trades"]),
        "trades (one per signal day)": int((daily.loc[trade_signal_days, "exit_day"] <= DEV_LAST_EXIT).sum()),
    })
updated_frequency = pd.DataFrame(rows).set_index("setup")
print(updated_frequency.to_string())

       signal days  trades (find_trades, superseded)  trades (one per signal day)
setup                                                                            
D1             103                                84                          103
D2             122                               110                          122


## Decision (pre-registration)

**Decision:** D1 and D2 are approved as written, with one rule change made before any outcome was computed.

**Rule change: every signal day is its own trade.**
- A signal at the close of day t buys at the open of t + 1 and sells at the open of t + 2, including on back-to-back signal days. Each trade pays the full 0.30% round trip.
- **Reason:** one-day trades on consecutive days do not overlap. A trade exits at the same open where the next one enters, so ignoring the second signal, as `find_trades(signal, 1)` does, would drop trades for no reason.
- Trades are built with the new tested function `find_one_day_trades` in `cryptosense.backtest`. `find_trades` is unchanged, because the Phase 6 backtests depend on it.
- This replaces every reference to `find_trades(signal, 1)` in the common rules and evaluation plan above. Everything else in the evaluation plan is unchanged.

**ETH/SOL threshold rule.** For the replication, ETH/USDT and SOL/USDT use their own development-period 2/3 quantile of `rv_open`, computed from their own data up to 2023-12-31 before any outcome, as proposed in the evaluation plan.

**Updated development frequency counts** (counts only, no returns; trades exiting on or before 2023-12-31):

| Setup | Signal days | Trades with `find_trades` (superseded) | Trades, one per signal day |
|---|---|---|---|
| D1 | 103 | 84 | 103 |
| D2 | 122 | 110 | 122 |

Nothing has been evaluated. The BTC development evaluation is the next step, after a separate instruction.

## Decision (assets)

**Decision:** BTC/USDT remains the primary asset, as pre-registered. SOL/USDT is evaluated as a secondary asset with unchanged rules and its own development threshold (rv_open 2/3 quantile 0.021055; development trades: D1 69, D2 64). A pooled BTC + SOL test is added: each asset uses its own threshold and its own base rates; on days when both assets trade, their net returns are averaged into one observation, because SOL largely moves with BTC and same-day trades are not independent. SOL's preparation (notebook 16) showed its most active hour is 10:00 New York in both periods, matching the US-open window. Known limitation: with about 65 development trades and a typical daily move of roughly 6–7%, SOL alone can only detect very large edges. No one-day outcome for any asset has been computed at the time of this decision.